In [6]:
import os
import sys

import pandas as pd
import numpy as np

sys.path.append("..")
from dataset_util import dataframes2objects, create_yolo_dataset

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# base_path = "../../../db/object-detection"
# base_path = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
base_path = "../../../../data/ObjectDetectionV3_1"
# base_path = "database"
df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df.head()

,label,not_ground,truncated,occluded,occupied_by_person,occupied,ID,xtl,ytl,xbr,ybr,x,y,img_id,id,corners
0,monitor,True,True,False,False,False,5.0,145.61,420.30,325.36,478.52,233.21,457.60,bdd640fb-0667-4ad1-9c80-317fa3b1799d,23b8c1e9-3924-46de-beb1-3b9046685257,NaN
1,chair,False,False,True,False,False,4.0,306.57,316.82,390.56,422.84,343.68,365.22,bdd640fb-0667-4ad1-9c80-317fa3b1799d,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,NaN
2,chair,False,False,True,False,False,3.0,198.15,225.50,284.59,299.78,238.68,275.36,bdd640fb-0667-4ad1-9c80-317fa3b1799d,972a8469-1641-4f82-8b9d-2434e465e150,NaN
3,chair,False,False,True,False,False,2.0,127.83,309.99,215.73,384.26,177.15,342.27,bdd640fb-0667-4ad1-9c80-317fa3b1799d,9a1de644-815e-46d1-bb8f-aa1837f8a88b,NaN
4,table,False,False,False,False,False,1.0,214.64,298.35,305.78,425.78,261.06,364.18,bdd640fb-0667-4ad1-9c80-317fa3b1799d,17fc695a-07a0-4a6e-8822-e8f36c031199,"[299.8, 306.5, 218.0, 306.1, 303.88, 421.14, 2..."


In [8]:
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))
df_imgs.head()

,id,name,width,height,batch,folder,task_id,job_id,job_url,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam2_H255_discaunt_29012024_2024-02-06-11h11m0...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
1,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
2,1a2a73ed-562b-4f79-8374-59eef50bea63,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
3,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0
4,29a3b2e9-5d65-4441-9588-42dea2bc372f,Cam2_H255_discaunt_29012024_2024-02-06-11h11m2...,640,480,batchai1-11.7,1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182,291e1aa6-1882-4eed-a749-ac5860383fa0


In [9]:
def f(row):
    scale_coef = 0.1
    w = abs(row["xbr"] - row["xtl"]) * scale_coef
    h = abs(row["ytl"] - row["ybr"]) * scale_coef
    
    x = row["x"]
    y = row["y"]
    corners = [
        x, y - h,
        x + w, y,
        x, y + h,
        x - w, y
    ]
    return str(corners)

In [10]:
# only two classes
dataset_name = "table_chair_v3"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

popular_labels = ["chair", "table"]
df = df[df["label"].isin(popular_labels)]

df = df[~(df['corners'].isna() & (df["label"] == "table"))]

imgs_ids_to_rm = np.concatenate([df[df['xtl'].isna()]["img_id"].values, df[df['x'].isna()]["img_id"].values])

df = df[~df["img_id"].isin(imgs_ids_to_rm)]
df_imgs = df_imgs[~df_imgs["id"].isin(imgs_ids_to_rm)]

print(df["label"].value_counts())

df.loc[df["label"] == "chair", "corners"] = df[df["label"] == "chair"].apply(f, axis=1)

objects, label_mapping = dataframes2objects(df, df_imgs, labels=popular_labels)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "..", "datasets", "v3_1", dataset_name), 
    os.path.join(base_path, "train_test_split.json"),
    kpt_shape=[5, 2]
)

label
chair    6946
table    2612
Name: count, dtype: int64
593 78


train labels: 593it [00:00, 3329.20it/s]
val labels: 78it [00:00, 51987.24it/s]
train images: 593it [00:00, 4302.35it/s]
val images: 78it [00:00, 5321.42it/s]
